Model Architecture.
We employ a Faster R-CNN two-stage object detector consisting of a Region Proposal Network (RPN) followed by a region-based classification and regression head. To improve detection performance for small-scale human instances commonly observed in satellite imagery, a ResNet-50 backbone with a Feature Pyramid Network (FPN) is adopted. The FPN enables multi-scale feature extraction, which is critical for handling objects with large scale variations.


!pip install -U numpy==1.26.4 scipy==1.11.4 torchmetrics==1.3.2

!pip uninstall -y numpy scipy 
!pip install --no-cache-dir numpy==1.26.4 scipy==1.11.4

In [ ]:
# =========================================
# CELL 1 — Imports
# =========================================
import os
import shutil
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.rpn import AnchorGenerator

from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
import os
import shutil
import random

# -------------------------------
# Define paths to the C2A dataset
# -------------------------------
base_path = "/kaggle/input/c2a-dataset/C2A_Dataset/new_dataset3"

train_img_dir = os.path.join(base_path, "train", "images")
train_lbl_dir = os.path.join(base_path, "train", "labels")
val_img_dir   = os.path.join(base_path, "val", "images")
val_lbl_dir   = os.path.join(base_path, "val", "labels")

# ------------------------------------------
# Create working directory (train/val/test)
# ------------------------------------------
work_dir = "/kaggle/working/flood_yolo"

for subdir in [
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val", "labels/test"
]:
    os.makedirs(os.path.join(work_dir, subdir), exist_ok=True)

In [ ]:
# -------------------------------
# Filter flood images
# -------------------------------
flood_train_imgs = [
    f for f in os.listdir(train_img_dir)
    if f.startswith("flood_image")
]

flood_val_imgs = [
    f for f in os.listdir(val_img_dir)
    if f.startswith("flood_image")
]

print(
    f"Found {len(flood_train_imgs)} flood training images and "
    f"{len(flood_val_imgs)} flood validation images."
)


# ------------------------------------------
# Split validation → val (80%) + test (20%)
# ------------------------------------------
random.seed(42)  # reproducibility

num_test = max(1, int(0.20 * len(flood_val_imgs)))
test_imgs = random.sample(flood_val_imgs, num_test)
val_imgs_final = list(set(flood_val_imgs) - set(test_imgs))

print(f"Validation images: {len(val_imgs_final)}")
print(f"Test images: {len(test_imgs)}")

# -------------------------------
# Copy training images
# -------------------------------
for img_file in flood_train_imgs:
    src_img = os.path.join(train_img_dir, img_file)
    src_lbl = os.path.join(train_lbl_dir, img_file.replace(".png", ".txt"))

    dst_img = os.path.join(work_dir, "images/train", img_file)
    dst_lbl = os.path.join(work_dir, "labels/train", img_file.replace(".png", ".txt"))

    shutil.copy(src_img, dst_img)
    shutil.copy(src_lbl, dst_lbl)

# -------------------------------
# Copy validation images (80%)
# -------------------------------
for img_file in val_imgs_final:
    src_img = os.path.join(val_img_dir, img_file)
    src_lbl = os.path.join(val_lbl_dir, img_file.replace(".png", ".txt"))

    dst_img = os.path.join(work_dir, "images/val", img_file)
    dst_lbl = os.path.join(work_dir, "labels/val", img_file.replace(".png", ".txt"))

    shutil.copy(src_img, dst_img)
    shutil.copy(src_lbl, dst_lbl)

# -------------------------------
# Copy test images (20%)
# -------------------------------
for img_file in test_imgs:
    src_img = os.path.join(val_img_dir, img_file)
    src_lbl = os.path.join(val_lbl_dir, img_file.replace(".png", ".txt"))

    dst_img = os.path.join(work_dir, "images/test", img_file)
    dst_lbl = os.path.join(work_dir, "labels/test", img_file.replace(".png", ".txt"))

    shutil.copy(src_img, dst_img)
    shutil.copy(src_lbl, dst_lbl)

In [ ]:
# -------------------------------
# Create dataset YAML
# -------------------------------
dataset_yaml = f"""
path: {work_dir}
train: images/train
val: images/val
test: images/test
names:
  0: human
"""

yaml_path = os.path.join(work_dir, "flood.yaml")
with open(yaml_path, "w") as f:
    f.write(dataset_yaml)

print("Flood-only dataset YAML created at", yaml_path)

In [ ]:
# =========================================
# CELL 4 — Dataset (YOLO txt -> FasterRCNN targets)
#   IMPORTANT changes:
#   - Return torchvision-style target keys: "boxes", "labels"
#   - If no GT: boxes shape (0,4), labels shape (0,)
#   - No dummy background boxes (prevents NaNs)
# =========================================
class FloodDataset(Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.image_dir, img_name)
        lbl_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + ".txt")

        # Load image
        raw_pil = Image.open(img_path).convert("RGB")
        w, h = raw_pil.size

        # To tensor [0,1]
        image_tensor = F.to_tensor(raw_pil)  # shape [3,H,W]

        boxes = []
        labels = []

        # Read YOLO labels (cls cx cy bw bh) normalized
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)

                    # ignore invalid sizes
                    if bw <= 0 or bh <= 0:
                        continue

                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h

                    # clamp
                    x1 = max(0.0, min(x1, w - 1))
                    y1 = max(0.0, min(y1, h - 1))
                    x2 = max(0.0, min(x2, w))
                    y2 = max(0.0, min(y2, h))

                    # filter degenerate
                    if x2 <= x1 or y2 <= y1:
                        continue

                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)  # human class = 1 (background is implicit)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,     # (N,4)
            "labels": labels,   # (N,)
            "image_id": torch.tensor([idx], dtype=torch.int64)
        }

        return image_tensor, target, raw_pil

In [ ]:
# =========================================
# CELL 5 — Dataloaders (torchvision expects lists)
# =========================================
def collate_fn(batch):
    images, targets, raws = [], [], []
    for img, tgt, raw in batch:
        images.append(img)
        targets.append(tgt)
        raws.append(raw)
    return images, targets, raws

base = "/kaggle/working/flood_yolo"

train_loader = DataLoader(
    FloodDataset(f"{base}/images/train", f"{base}/labels/train"),
    batch_size=2,  # you can keep 1 if GPU is small
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    FloodDataset(f"{base}/images/val", f"{base}/labels/val"),
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    FloodDataset(f"{base}/images/test", f"{base}/labels/test"),
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)


In [ ]:
# =========================================
# CELL 6 — Build torchvision Faster R-CNN (ResNet50-FPN)
# Anchor sizes tuned for small UAV humans (includes smaller anchors)
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = resnet_fpn_backbone(
    backbone_name="resnet50",
    weights="IMAGENET1K_V1",
    trainable_layers=3
)

# FPN has 5 levels; provide sizes per level (tuple of tuples)
anchor_generator = AnchorGenerator(
    sizes=((16,), (32,), (64,), (128,), (256,)),
    aspect_ratios=((0.5, 1.0, 2.0),) * 5
)

model = FasterRCNN(
    backbone=backbone,
    num_classes=2,  # 0=background, 1=human
    rpn_anchor_generator=anchor_generator
).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0025,
    momentum=0.9,
    weight_decay=0.0005
)

In [ ]:
# =========================================
# CELL 7 — Training loop (stable: checks loss BEFORE backward)
# Saves best checkpoint by validation mAP (map_50)
# =========================================
def train_one_epoch(model, loader, optimizer, device, grad_clip=5.0):
    model.train()
    total_loss = 0.0
    num_steps = 0

    for images, targets, _ in loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            print("⚠️ Non-finite loss. Skipping batch. Losses:",
                  {k: float(v.detach().cpu()) for k, v in loss_dict.items()})
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        total_loss += float(loss.detach().cpu())
        num_steps += 1

    return total_loss / max(1, num_steps)


@torch.no_grad()
def evaluate_map(model, loader, device):
    metric = MeanAveragePrecision(iou_type="bbox")
    model.eval()

    for images, targets, _ in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        preds = []
        gts = []
        for out, tgt in zip(outputs, targets):
            preds.append({
                "boxes": out["boxes"].detach().cpu(),
                "scores": out["scores"].detach().cpu(),
                "labels": out["labels"].detach().cpu(),
            })
            gts.append({
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu(),
            })

        metric.update(preds, gts)

    return metric.compute()


@torch.no_grad()
def compute_precision_recall(model, loader, device, iou_thresh=0.5, score_thresh=0.5):
    model.eval()
    TP, FP, FN = 0, 0, 0

    for images, targets, _ in loader:
        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            gt_boxes = tgt["boxes"].to(device)

            pred_boxes = out["boxes"].to(device)
            scores = out["scores"].to(device)

            keep = scores >= score_thresh
            pred_boxes = pred_boxes[keep]

            # No GT
            if gt_boxes.numel() == 0:
                FP += pred_boxes.shape[0]
                continue

            # No predictions
            if pred_boxes.shape[0] == 0:
                FN += gt_boxes.shape[0]
                continue

            ious = torchvision.ops.box_iou(pred_boxes, gt_boxes)
            matched_gt = set()

            for i in range(pred_boxes.shape[0]):
                max_iou, gt_idx = ious[i].max(dim=0)
                gi = int(gt_idx.item())
                if float(max_iou) >= iou_thresh and gi not in matched_gt:
                    TP += 1
                    matched_gt.add(gi)
                else:
                    FP += 1

            FN += gt_boxes.shape[0] - len(matched_gt)

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)
    return precision, recall


# Train
ckpt_dir = "/kaggle/working/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

num_epochs = 100  
best_map50 = -1.0

history = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate_map(model, val_loader, device)
    val_map50 = float(val_metrics["map_50"])

    if val_map50 > best_map50:
        best_map50 = val_map50
        torch.save(model.state_dict(), f"{ckpt_dir}/fasterrcnn_best.pth")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_map": float(val_metrics["map"]),
        "val_map_50": float(val_metrics["map_50"]),
        "val_map_75": float(val_metrics["map_75"]),
        "val_mar_100": float(val_metrics["mar_100"]),
    })

    print(
        f"Epoch {epoch+1:03d} | "
        f"train_loss={train_loss:.4f} | val_map50={val_map50:.4f} | best_map50={best_map50:.4f}"
    )

In [ ]:
# =========================================
# CELL 8 — Final evaluation on val & test + save EXCEL (keeps your Excel saving idea)
# =========================================
# Load best checkpoint for reporting
ckpt_dir = "/kaggle/working/checkpoints"
best_path = f"{ckpt_dir}/fasterrcnn_best.pth"
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print("Loaded best checkpoint:", best_path)

val_metrics = evaluate_map(model, val_loader, device)
test_metrics = evaluate_map(model, test_loader, device)

val_prec, val_rec = compute_precision_recall(model, val_loader, device, iou_thresh=0.5, score_thresh=0.5)
test_prec, test_rec = compute_precision_recall(model, test_loader, device, iou_thresh=0.5, score_thresh=0.5)

df = pd.DataFrame([
    ["Validation",
     float(val_metrics["map"]), float(val_metrics["map_50"]), float(val_metrics["map_75"]), float(val_metrics["mar_100"]),
     float(val_prec), float(val_rec)],
    ["Test",
     float(test_metrics["map"]), float(test_metrics["map_50"]), float(test_metrics["map_75"]), float(test_metrics["mar_100"]),
     float(test_prec), float(test_rec)],
], columns=[
    "Split",
    "mAP@0.5:0.95",
    "mAP@0.5",
    "mAP@0.75",
    "mAR@100",
    "Precision@0.5",
    "Recall@0.5"
])

out_xlsx = "/kaggle/working/faster_rcnn_metrics.xlsx"
df.to_excel(out_xlsx, index=False)
print("Saved:", out_xlsx)

# Also save training history (optional, helpful for paper)
hist_xlsx = "/kaggle/working/faster_rcnn_training_history.xlsx"
pd.DataFrame(history).to_excel(hist_xlsx, index=False)
print("Saved:", hist_xlsx)

In [ ]:
# =========================================
# CELL 9 — Save first 20 test detections as PNG (same idea as your notebook)
# =========================================
save_dir = "/kaggle/working/test_detections"
os.makedirs(save_dir, exist_ok=True)

model.eval()
count = 0

with torch.no_grad():
    for images, _, raws in test_loader:
        if count >= 20:
            break

        images = [img.to(device) for img in images]
        outputs = model(images)

        for out, raw in zip(outputs, raws):
            if count >= 20:
                break

            draw = ImageDraw.Draw(raw)
            boxes = out["boxes"].detach().cpu().numpy()
            scores = out["scores"].detach().cpu().numpy()

            for box, score in zip(boxes, scores):
                if score < 0.5:
                    continue
                x1, y1, x2, y2 = box.tolist()
                draw.rectangle([x1, y1, x2, y2], outline="red", width=2)

            raw.save(f"{save_dir}/test_{count}.png")
            count += 1

print("Saved detections to:", save_dir)

In [ ]:
# =========================================
# CELL 10 — Zip detections folder (same as your notebook)
# =========================================
output_path = "/kaggle/working/working_directory"
shutil.make_archive(output_path, 'zip', save_dir)
print("Zipped successfully:", output_path + ".zip")


In [ ]:
# ============================================================
# FULL CODE: Save first 20 test predictions with YOLO-like style
# (Blue boxes + "human 0.xx" label with filled blue background)
# Works with torchvision Faster R-CNN outputs
# ============================================================

import os
import cv2
import torch
import numpy as np

# ---------- Settings ----------
SAVE_DIR = os.path.join(work_dir, "predictions", "test_yolo_style")  # keep your work_dir naming
os.makedirs(SAVE_DIR, exist_ok=True)

SCORE_THRESH = 0.25   # like your YOLO conf=0.25
MAX_IMAGES = 20

# YOLO-like style (OpenCV)
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7      # good default for UAV images (you can tweak to 0.6/0.8)
THICKNESS = 1
BOX_THICKNESS = 2
COLOR_BOX = (255, 0, 0)      # Blue in BGR
COLOR_TEXT = (255, 255, 255) # White
COLOR_FILL = (255, 0, 0)     # Blue fill in BGR

# ---------- Helper: draw one labeled box ----------
def draw_yolo_style_box(img_bgr, box_xyxy, score, label_name="human"):
    """
    img_bgr: np.ndarray (H,W,3) BGR
    box_xyxy: iterable [x1,y1,x2,y2] float/int
    score: float
    """
    h, w = img_bgr.shape[:2]
    x1, y1, x2, y2 = box_xyxy

    # Clamp + int
    x1 = int(max(0, min(x1, w - 1)))
    y1 = int(max(0, min(y1, h - 1)))
    x2 = int(max(0, min(x2, w - 1)))
    y2 = int(max(0, min(y2, h - 1)))

    # Draw bounding box
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), COLOR_BOX, BOX_THICKNESS)

    # Label text
    text = f"{label_name} {score:.2f}"

    # Text size
    (tw, th), baseline = cv2.getTextSize(text, FONT, FONT_SCALE, THICKNESS)

    # Place label above the box if possible, else inside
    y_text_top = y1 - th - baseline - 6
    if y_text_top < 0:
        y_text_top = y1 + 2
    x_text_left = x1

    # Background rectangle (filled)
    cv2.rectangle(
        img_bgr,
        (x_text_left, y_text_top),
        (x_text_left + tw + 6, y_text_top + th + baseline + 6),
        COLOR_FILL,
        -1
    )

    # Put text
    cv2.putText(
        img_bgr,
        text,
        (x_text_left + 3, y_text_top + th + 3),
        FONT,
        FONT_SCALE,
        COLOR_TEXT,
        THICKNESS,
        cv2.LINE_AA
    )

    return img_bgr


# ---------- Main: run on test set and save first 20 ----------
model.eval()
count = 0

with torch.no_grad():
    for images, targets, raws in test_loader:
        if count >= MAX_IMAGES:
            break

        # Forward
        images_gpu = [img.to(device) for img in images]
        outputs = model(images_gpu)

        for out, raw_pil in zip(outputs, raws):
            if count >= MAX_IMAGES:
                break

            # Convert PIL -> BGR for OpenCV drawing
            img_rgb = np.array(raw_pil)                 # (H,W,3) RGB
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

            boxes = out["boxes"].detach().cpu().numpy()
            scores = out["scores"].detach().cpu().numpy()
            labels = out["labels"].detach().cpu().numpy()

            # Draw predictions
            for box, score, lbl in zip(boxes, scores, labels):
                if float(score) < SCORE_THRESH:
                    continue
                # Since you have only human class, label is always 1
                img_bgr = draw_yolo_style_box(img_bgr, box, float(score), label_name="human")

            # Save outputs like your YOLO notebook
            base_name = f"test_{count:03d}"
            png_path = os.path.join(SAVE_DIR, f"{base_name}.png")
            eps_path = os.path.join(SAVE_DIR, f"{base_name}.eps")

            # Save PNG
            cv2.imwrite(png_path, img_bgr)

            # Save EPS (Matplotlib is easiest for EPS)
            # EPS is vector-like container; image itself will still be raster, but journals accept this.
            import matplotlib.pyplot as plt
            img_rgb_out = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(6, 6))
            plt.imshow(img_rgb_out)
            plt.axis("off")
            plt.savefig(eps_path, format="eps", bbox_inches="tight")
            plt.close()

            print(f"[{count+1}/{MAX_IMAGES}] Saved: {png_path} and {eps_path}")
            count += 1

print("Done. Saved YOLO-style predictions to:", SAVE_DIR)


In [ ]:
import shutil

# Path to the folder to zip
folder_path = "/kaggle/working/flood_yolo/predictions/test_yolo_style"

# Output zip file path (without .zip extension)
output_path = "/kaggle/working/test_images_fasterRCNN"

shutil.make_archive(output_path, 'zip', folder_path)

print("Zipped successfully!")